# GSE295514 - read RDS (R kernel)

Open `GSE295514_ALS_mouse_brain.rds` interactively, inspect its class / assays / meta.data, then export Python-readable intermediates to `data/intermediate_from_r/GSE295514/`:

* `counts.mtx` (genes x cells, Matrix Market)
* `metadata.csv`
* `genes.csv` (gene_symbol)
* `barcodes.csv` (cell_id)

The Python notebook 01 reads these via `io_dense.read_from_r_intermediate(...)`.

Requires an **R Jupyter kernel (IRkernel)** plus `Matrix`, and `Seurat`/`SeuratObject` or `SingleCellExperiment` depending on the object.

In [ ]:
library(Matrix)

rds_path <- "../../data/raw/GSE295514/GSE295514_ALS_mouse_brain.rds"
obj <- readRDS(rds_path)

class(obj)

In [ ]:
names(obj)
tryCatch(slotNames(obj), error = function(e) e$message)

## If Seurat

In [ ]:
# library(Seurat)
# Assays(obj)
# DefaultAssay(obj)
# head(obj@meta.data)
# colnames(obj@meta.data)

In [ ]:
# counts <- GetAssayData(obj, assay = DefaultAssay(obj), slot = "counts")
# dim(counts)
# counts[1:5, 1:5]

## If SingleCellExperiment

In [ ]:
# library(SingleCellExperiment)
# assayNames(obj)
# counts <- assay(obj, "counts")
# colData(obj)
# rowData(obj)

## Export intermediates for Python

Set `counts` (genes x cells), `meta` (cell metadata data.frame) from the branch above, then run:

In [ ]:
out_dir <- "../../data/intermediate_from_r/GSE295514"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# --- choose the right extraction for your object: ---
if (inherits(obj, "Seurat")) {
    counts <- GetAssayData(obj, assay = DefaultAssay(obj), slot = "counts")
    meta <- obj@meta.data
} else if (inherits(obj, "SingleCellExperiment")) {
    counts <- SummarizedExperiment::assay(obj, "counts")
    meta <- as.data.frame(SummarizedExperiment::colData(obj))
} else {
    stop(paste("Unsupported class:", paste(class(obj), collapse = ", ")))
}

counts <- as(counts, "CsparseMatrix")
writeMM(counts, file.path(out_dir, "counts.mtx"))
write.csv(meta, file.path(out_dir, "metadata.csv"))
write.csv(data.frame(gene_symbol = rownames(counts)),
          file.path(out_dir, "genes.csv"), row.names = FALSE)
write.csv(data.frame(cell_id = colnames(counts)),
          file.path(out_dir, "barcodes.csv"), row.names = FALSE)

cat("wrote intermediates to", out_dir, "\n")
dim(counts)

Now go to **notebooks/python/01_load_each_gse_to_anndata.ipynb**, GSE295514 section, which calls `io_dense.read_from_r_intermediate(...)`.